# OpenAI API Examples

Companion notebook for [`README.md`](./README.md).  
All examples use the `openai` library and load credentials from a `.env` file.

**Requirements:** `conda activate agents` and `pip install openai python-dotenv numpy`

In [ ]:
# Shared setup — run this cell first
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()          # reads OPENAI_API_KEY from .env
client = OpenAI()      # picks up the key automatically
print("Client ready.")

Client ready.


---
## 1. Basic Chat Completion

Every request is a list of `messages` with roles `system`, `user`, or `assistant`.

In [ ]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user",   "content": "What is the capital of Japan?"},
    ],
)

print(response.choices[0].message.content)
print(f"\nTokens used: {response.usage.total_tokens}")

The capital of Japan is Tokyo.

Tokens used: 31


---
## 2. Multi-Turn Conversation

Maintain context by appending each assistant reply to the `messages` list.

In [3]:
messages = [{"role": "system", "content": "You are a helpful assistant."}]

def chat(user_input: str) -> str:
    messages.append({"role": "user", "content": user_input})
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
    )
    reply = response.choices[0].message.content
    messages.append({"role": "assistant", "content": reply})
    return reply

print(chat("My name is Alice."))
print()
print(chat("What is my name?"))   # model should remember

Hello, Alice! How can I assist you today?

Your name is Alice. How can I help you today?


---
## 3. Streaming

### 3a. Simple `stream=True`

Receive tokens incrementally as they are generated.

In [4]:
stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Write a haiku about Python."}],
    stream=True,
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
print()

Serpent in the code,  
Logic flows like gentle streams,  
Silk threads weave the dream.


### 3b. `.stream()` context manager

Richer event types; access to the final completion object with token usage.

In [5]:
with client.chat.completions.stream(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "List three benefits of Python."}],
    stream_options={"include_usage": True},
) as stream:
    for event in stream:
        if event.type == "content.delta":
            print(event.delta, end="", flush=True)

completion = stream.get_final_completion()
print(f"\n\nTotal tokens: {completion.usage.total_tokens}")

Certainly! Here are three benefits of Python:

1. **Readability and Simplicity**: Python's syntax is designed to be clear and intuitive, making it easy for beginners to learn and for experienced developers to read and maintain code. This readability promotes better collaboration and reduces the cognitive load when working on complex projects.

2. **Versatility and Flexibility**: Python is a versatile language that supports multiple programming paradigms, including procedural, object-oriented, and functional programming. It can be used for a wide range of applications, from web development and data analysis to artificial intelligence and scientific computing.

3. **Large Standard Library and Community Support**: Python has a comprehensive standard library that provides many modules and functions for various tasks, which reduces the need to write code from scratch. Additionally, Python has a large and active community, making it easy to find libraries, frameworks, and resources, as well 

---
## 4. Structured Output with Pydantic

Use `.parse()` with a Pydantic model as `response_format`.  
The SDK converts it to JSON schema, sends it to the API, and parses the reply back into typed Python objects.

> Requires `gpt-4o-2024-08-06` or later.

In [6]:
from typing import List
from pydantic import BaseModel

class Step(BaseModel):
    explanation: str
    output: str

class MathResponse(BaseModel):
    steps: List[Step]
    final_answer: str

completion = client.chat.completions.parse(
    model="gpt-4o-2024-08-06",
    messages=[
        {"role": "system", "content": "You are a math tutor."},
        {"role": "user",   "content": "Solve: 8x + 31 = 2"},
    ],
    response_format=MathResponse,
)

message = completion.choices[0].message
if message.parsed:
    for step in message.parsed.steps:
        print(f"{step.explanation}  →  {step.output}")
    print("\nAnswer:", message.parsed.final_answer)
else:
    print("Refusal:", message.refusal)

Start solving the equation by isolating the term with x. Begin by subtracting 31 from both sides to move the constant term to the other side of the equation.  →  8x + 31 - 31 = 2 - 31
Simplify both sides of the equation. On the left side, 31 cancels out, leaving only 8x. On the right side, calculate 2 - 31.  →  8x = -29
Now, solve for x by dividing both sides by 8 to isolate x.  →  x = -29 / 8

Answer: x = -\frac{29}{8}


---
## 5. Tool Use / Function Calling

1. Define tool schemas.  
2. First call: model decides whether to call a tool.  
3. Execute the function locally.  
4. Second call: model produces the final answer using the tool result.

In [7]:
import json

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Return current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"},
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                },
                "required": ["city"],
            },
        },
    }
]

messages = [{"role": "user", "content": "What's the weather in Tokyo?"}]

# First call
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools,
    tool_choice="auto",
)

message = response.choices[0].message
print("Finish reason:", response.choices[0].finish_reason)

if message.tool_calls:
    tool_call = message.tool_calls[0]
    args = json.loads(tool_call.function.arguments)
    print("Tool called:", tool_call.function.name, "|", args)

    # Simulate the actual function
    tool_result = {"temperature": 18, "condition": "Cloudy", "city": args["city"]}

    messages.append(message)
    messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": json.dumps(tool_result),
    })

    # Second call — final answer
    final = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools,
    )
    print("\nFinal answer:", final.choices[0].message.content)

Finish reason: tool_calls
Tool called: get_weather | {'city': 'Tokyo'}

Final answer: The weather in Tokyo is currently 18°C and cloudy.


---
## 6. Vision (Image Input)

Pass images alongside text in `content`. Supports public URLs and base64 local files.

### 6a. Image from URL

In [8]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text",      "text": "What animal is in this image?"},
                {"type": "image_url", "image_url": {
                    "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/d/d5/2023_06_08_Raccoon1.jpg/400px-2023_06_08_Raccoon1.jpg"
                }},
            ],
        }
    ],
)
print(response.choices[0].message.content)

The animal in the image is a raccoon.


### 6b. Image from local file (base64)

In [9]:
import base64

# Replace with a real local image path to test
image_path = "./assets/2023_06_08_Raccoon1.jpg"
try:
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text",      "text": "Describe this image."},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
                ],
            }
        ],
    )
    print(response.choices[0].message.content)
except FileNotFoundError:
    print(f"File not found: {image_path} — replace with a real image path to test.")

The image features a raccoon peeking around the side of a tree trunk. The tree's bark is textured and rough, showcasing its natural patterns. The background appears dark, which helps highlight the raccoon's distinctive facial markings and fur. The raccoon's curious expression adds a sense of playfulness to the scene.


---
## 7. Embeddings

Convert text to a numerical vector for semantic search, clustering, or RAG.

In [10]:
import numpy as np

# Single embedding
response = client.embeddings.create(
    model="text-embedding-3-small",
    input="The quick brown fox jumps over the lazy dog.",
)
vector = response.data[0].embedding
print(f"Dimensions: {len(vector)}")

Dimensions: 1536


In [11]:
# Batch embeddings + cosine similarity
texts = [
    "Machine learning is a branch of AI.",
    "Deep learning uses neural networks.",
    "I love pizza and pasta.",
]

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=texts,
)
vectors = [np.array(item.embedding) for item in response.data]

def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print(f"ML vs DL:    {cosine_similarity(vectors[0], vectors[1]):.3f}")
print(f"ML vs Food:  {cosine_similarity(vectors[0], vectors[2]):.3f}")

ML vs DL:    0.497
ML vs Food:  0.081


---
## 8. Async Client

Use `AsyncOpenAI` for non-blocking calls in async frameworks (FastAPI, asyncio).

> In a Jupyter notebook `asyncio.run()` is not needed — use `await` directly.

In [12]:
from openai import AsyncOpenAI

async_client = AsyncOpenAI()   # also reads OPENAI_API_KEY from env

# Basic async completion (await works directly in Jupyter)
response = await async_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Name three planets."}],
)
print(response.choices[0].message.content)

Mercury, Venus, and Mars.


In [16]:
# Async streaming
stream = await async_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Count to five, one word per line."}],
    stream=True,
)

async for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
print()

One  
Two  
Three  
Four  
Five  


---
## 9. Audio and Speech

### 9a. Text-to-Speech (TTS)

Stream synthesis directly to a file to avoid loading the full response into memory.  
Voices: `alloy`, `echo`, `fable`, `onyx`, `nova`, `shimmer`. Models: `tts-1` (fast) / `tts-1-hd` (higher quality).

In [18]:
from pathlib import Path

speech_file = Path("speech.mp3")

with client.audio.speech.with_streaming_response.create(
    model="tts-1",
    voice="alloy",
    input="The quick brown fox jumped over the lazy dogs.",
) as response:
    response.stream_to_file(speech_file)

print(f"Saved to {speech_file}  ({speech_file.stat().st_size:,} bytes)")

SyntaxError: invalid syntax (1886648238.py, line 1)

### 9b. Transcription (Speech-to-Text)

Transcribe the audio file we just created. Use `response_format="verbose_json"` for word-level timestamps.

In [ ]:
# Basic transcription
with open(speech_file, "rb") as f:
    transcription = client.audio.transcriptions.create(
        model="whisper-1",
        file=f,
        response_format="text",
    )
print("Transcription:", transcription)

# With word-level timestamps
with open(speech_file, "rb") as f:
    verbose = client.audio.transcriptions.create(
        model="whisper-1",
        file=f,
        response_format="verbose_json",
        timestamp_granularities=["word"],
    )
print("\nWord timestamps:")
for word in verbose.words:
    print(f"  {word.word:20s} {word.start:.2f}s – {word.end:.2f}s")

---
## 10. Responses API

OpenAI's newer, simpler API. Key differences from Chat Completions:
- `input` (string or list) replaces `messages`; `instructions` replaces the system message
- Built-in server-side tools: `web_search_preview`, `code_interpreter`
- **Stateful**: pass `previous_response_id` instead of resending history
- **Background mode**: pause and resume long-running generations

### 10a. Basic call

In [ ]:
response = client.responses.create(
    model="gpt-4o-mini",
    instructions="You are a concise assistant.",
    input="What is the capital of Japan?",
)
print(response.output_text)

### 10b. Stateful multi-turn

Pass `previous_response_id` — the server remembers the prior turn, no need to resend history.

In [ ]:
r1 = client.responses.create(model="gpt-4o-mini", input="My name is Alice.")
r2 = client.responses.create(
    model="gpt-4o-mini",
    input="What is my name?",
    previous_response_id=r1.id,
)
print(r2.output_text)   # "Your name is Alice."

### 10c. Streaming

In [ ]:
with client.responses.stream(
    model="gpt-4o-mini",
    input="Write a haiku about Python.",
) as stream:
    for event in stream:
        if event.type == "response.output_text.delta":
            print(event.delta, end="", flush=True)

final = stream.get_final_response()
print(f"\n\nTotal tokens: {final.usage.total_tokens}")

### 10d. Structured output with `responses.parse()`

In [ ]:
from typing import List
from pydantic import BaseModel

class Step(BaseModel):
    explanation: str
    output: str

class MathResponse(BaseModel):
    steps: List[Step]
    final_answer: str

rsp = client.responses.parse(
    model="gpt-4o-2024-08-06",
    input="Solve: 8x + 31 = 2",
    text_format=MathResponse,
)

parsed = rsp.output[0].content[0].parsed
for step in parsed.steps:
    print(f"{step.explanation}  →  {step.output}")
print("Answer:", parsed.final_answer)

### 10e. Built-in server-side tools

`web_search_preview` and `code_interpreter` run on OpenAI's infrastructure — no execution code on your side.

In [ ]:
# Web search (server-side, no execution needed)
response = client.responses.create(
    model="gpt-4o-mini",
    tools=[{"type": "web_search_preview"}],
    input="What are the top Python news stories this week?",
)
print(response.output_text)

In [ ]:
# Code interpreter (server-side execution)
response = client.responses.create(
    model="gpt-4o-mini",
    tools=[{"type": "code_interpreter", "container": {"type": "auto"}}],
    input="Calculate the square root of 273 * 312821 + 1782. Show the Python code and the numeric answer.",
)
print(response.output_text)

### 10f. Vision with the Responses API

In [ ]:
response = client.responses.create(
    model="gpt-4o-mini",
    input=[{
        "role": "user",
        "content": [
            {"type": "input_text",  "text": "What animal is in this image?"},
            {"type": "input_image", "image_url": "https://upload.wikimedia.org/wikipedia/commons/thumb/d/d5/2023_06_08_Raccoon1.jpg/400px-2023_06_08_Raccoon1.jpg"},
        ],
    }],
)
print(response.output_text)

---
## 11. MCP Connectors

The OpenAI Agents SDK provides three MCP transport classes:

| Class | Transport | When to use |
|---|---|---|
| `MCPServerStdio` | Local subprocess stdin/stdout | Local servers (e.g. `npx`, `uvx`) |
| `MCPServerSse` | HTTP + Server-Sent Events | Remote SSE servers |
| `MCPServerStreamableHttp` | Bidirectional HTTP streaming | Remote Streamable HTTP servers |

```bash
pip install openai-agents
```

> All three are used as **async context managers** inside the Agents SDK — see section 12 for `Agent` and `Runner`.

### 11a. Local MCP server via stdio

In [ ]:
import asyncio
from agents import Agent, Runner
from agents.mcp import MCPServerStdio

# Requires Node.js and the @modelcontextprotocol/server-filesystem package
# npm install -g @modelcontextprotocol/server-filesystem

async def run_filesystem_agent():
    async with MCPServerStdio(
        params={
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-filesystem", "/tmp"],
        },
        cache_tools_list=True,
    ) as mcp_server:
        agent = Agent(
            name="File Assistant",
            instructions="Use the filesystem tools to help the user.",
            mcp_servers=[mcp_server],
        )
        result = await Runner.run(agent, "List the files in /tmp.")
        print(result.final_output)

# asyncio.run(run_filesystem_agent())   # uncomment to run
print("MCPServerStdio example ready. Requires Node.js + @modelcontextprotocol/server-filesystem.")

### 11b. Remote MCP server via SSE and Streamable HTTP

Replace the URL and token with your actual server details.

In [ ]:
from agents.mcp import MCPServerSse, MCPServerStreamableHttp

# --- SSE (Server-Sent Events) ---
async def run_sse_agent():
    async with MCPServerSse(
        params={
            "url": "https://my-mcp-server.example.com/sse",
            "headers": {"Authorization": "Bearer MY_TOKEN"},
        },
        cache_tools_list=True,
    ) as mcp_server:
        agent = Agent(
            name="Remote Agent",
            instructions="Use tools from the MCP server.",
            mcp_servers=[mcp_server],
        )
        result = await Runner.run(agent, "What tools do you have?")
        print(result.final_output)

# --- Streamable HTTP ---
async def run_streamable_http_agent():
    async with MCPServerStreamableHttp(
        params={"url": "https://my-mcp-server.example.com/mcp"},
        cache_tools_list=True,
    ) as mcp_server:
        agent = Agent(
            name="HTTP MCP Agent",
            instructions="Use tools from the MCP server.",
            mcp_servers=[mcp_server],
        )
        result = await Runner.run(agent, "Run a calculation.")
        print(result.final_output)

print("SSE and StreamableHTTP examples defined. Replace URLs to run.")

---
## 12. Agents SDK

`openai-agents` is a lightweight framework for multi-step, multi-agent workflows.  
Core primitives: `Agent`, `Runner`, `@function_tool`, handoffs, guardrails, tracing.

> **Skills:** In the Agents SDK, reusable capabilities are expressed as `@function_tool` decorated functions — the equivalent of Codex CLI "skills". Bundle related tools in a module and import them into any agent.

### 12a. Basic agent

In [ ]:
from agents import Agent, Runner

agent = Agent(
    name="Assistant",
    instructions="You are a helpful, concise assistant.",
    model="gpt-4o-mini",
)

result = await Runner.run(agent, "What is the capital of Japan?")
print(result.final_output)

### 12b. `@function_tool` — skills as decorated functions

Type hints + docstring become the JSON schema automatically.

In [ ]:
from pydantic import BaseModel
from agents import function_tool

class Weather(BaseModel):
    city: str
    temperature_range: str
    conditions: str

@function_tool
def get_weather(city: str) -> Weather:
    """Get the current weather for a city.

    Args:
        city: The city name.
    """
    return Weather(city=city, temperature_range="14-20°C", conditions="Sunny")

weather_agent = Agent(
    name="Weather Agent",
    instructions="Use the weather tool to answer questions.",
    tools=[get_weather],
    model="gpt-4o-mini",
)

result = await Runner.run(weather_agent, "What's the weather in Tokyo?")
print(result.final_output)

### 12c. Handoffs — multi-agent routing

In [ ]:
history_agent = Agent(
    name="History Tutor",
    handoff_description="Specialist for historical questions.",
    instructions="Answer history questions clearly and concisely.",
    model="gpt-4o-mini",
)

math_agent = Agent(
    name="Math Tutor",
    handoff_description="Specialist for math questions.",
    instructions="Explain math step by step with worked examples.",
    model="gpt-4o-mini",
)

triage_agent = Agent(
    name="Triage Agent",
    instructions="Route each question to the right specialist.",
    handoffs=[history_agent, math_agent],
    model="gpt-4o-mini",
)

result = await Runner.run(triage_agent, "Who was the first US president?")
print(f"Answer: {result.final_output}")
print(f"Answered by: {result.last_agent.name}")

### 12d. Agent as a tool

Sub-agent returns a result without taking over the conversation.

In [ ]:
translator = Agent(
    name="Translator",
    instructions="Translate the given text to the requested target language. Return only the translation.",
    model="gpt-4o-mini",
)

orchestrator = Agent(
    name="Orchestrator",
    instructions="Coordinate translation tasks using the translate_text tool.",
    model="gpt-4o-mini",
    tools=[
        translator.as_tool(
            tool_name="translate_text",
            tool_description="Translate text to a given language.",
        ),
    ],
)

result = await Runner.run(orchestrator, "Translate 'Hello, world!' to Spanish, French, and Japanese.")
print(result.final_output)

### 12e. Streaming events

In [ ]:
import random
from agents import ItemHelpers

@function_tool
def how_many_jokes() -> int:
    """Return how many jokes to tell (1–4)."""
    return random.randint(1, 4)

joker = Agent(
    name="Joker",
    instructions="Call how_many_jokes, then tell exactly that many jokes.",
    tools=[how_many_jokes],
    model="gpt-4o-mini",
)

result = Runner.run_streamed(joker, "Tell me some jokes.")
async for event in result.stream_events():
    if event.type == "run_item_stream_event":
        item = event.item
        if item.type == "tool_call_item":
            print(f"[tool call] {item.raw_item.name}")
        elif item.type == "tool_call_output_item":
            print(f"[tool output] {item.output}")
        elif item.type == "message_output_item":
            print(f"[message]\n{ItemHelpers.text_message_output(item)}")

---
## 13. File Uploads and PDF Handling

OpenAI's Chat Completions and Responses APIs do **not** accept raw PDFs — you must parse the content first.

**Strategy:**
- **Text + structure** → extract with `pymupdf` (fitz); send as plain-text in the message
- **Images in the PDF** → extract as PNG with `pymupdf`; send as `image_url` (base64)
- **Tables** → extract with `pdfplumber` → Markdown table in the text message

```bash
pip install pymupdf pdfplumber
```

We first create a small synthetic PDF to use for all examples below.

In [ ]:
import fitz   # pymupdf

# Create a tiny synthetic PDF with text, a table, and an embedded image
def make_sample_pdf(path: str = "sample.pdf") -> str:
    doc = fitz.open()
    page = doc.new_page()

    # Text
    page.insert_text((50, 72), "Q3 Financial Report", fontsize=16, fontname="helv")
    page.insert_text((50, 100), "Revenue grew 12% year-over-year, driven by cloud services.", fontsize=11)
    page.insert_text((50, 120), "Operating margin improved from 18% to 21%.", fontsize=11)

    # Simple table drawn with lines + text
    table_data = [
        ["Division",   "Q3 2023", "Q3 2024"],
        ["Cloud",      "$120M",   "$145M"],
        ["Hardware",   "$80M",    "$78M"],
        ["Services",   "$55M",    "$62M"],
    ]
    x0, y0, col_w, row_h = 50, 160, 100, 20
    for r, row in enumerate(table_data):
        for c, cell in enumerate(row):
            page.insert_text((x0 + c * col_w + 4, y0 + r * row_h + 14), cell, fontsize=10)
        page.draw_rect(fitz.Rect(x0, y0 + r * row_h, x0 + 3 * col_w, y0 + (r + 1) * row_h))

    # Embed a tiny coloured rectangle as a "figure"
    page.draw_rect(fitz.Rect(50, 260, 200, 310), color=(0.2, 0.4, 0.8), fill=(0.6, 0.8, 1.0))
    page.insert_text((55, 325), "Figure 1: Revenue mix visualisation", fontsize=9)

    doc.save(path)
    doc.close()
    return path

pdf_path = make_sample_pdf()
print(f"Created {pdf_path}")

### 13a. Extract text and send to OpenAI

In [ ]:
def pdf_to_text(path: str) -> str:
    """Extract all text from a PDF, page by page."""
    doc = fitz.open(path)
    pages = [f"--- Page {i+1} ---\n{page.get_text()}" for i, page in enumerate(doc)]
    doc.close()
    return "\n\n".join(pages)

text = pdf_to_text(pdf_path)
print(text)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a document analyst."},
        {"role": "user",   "content": f"Summarise this document in two sentences:\n\n{text}"},
    ],
)
print("\nSummary:", response.choices[0].message.content)

### 13b. Extract images from a PDF and send as vision input

`pymupdf` renders each page as a PNG. This captures everything — raster images, charts drawn with vectors, diagrams — because it rasterises the full page instead of trying to extract individual image objects.

In [ ]:
import base64

def pdf_pages_as_images(path: str, dpi: int = 150) -> list[dict]:
    """Render each PDF page to a PNG and return base64-encoded images."""
    doc = fitz.open(path)
    images = []
    for i, page in enumerate(doc):
        mat = fitz.Matrix(dpi / 72, dpi / 72)   # scale factor
        pix = page.get_pixmap(matrix=mat)
        b64 = base64.b64encode(pix.tobytes("png")).decode()
        images.append({"page": i + 1, "mime": "image/png", "b64": b64})
    doc.close()
    return images

pages = pdf_pages_as_images(pdf_path)
print(f"Rendered {len(pages)} page(s)")

# Send page 1 to the vision model
img = pages[0]
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{
        "role": "user",
        "content": [
            {"type": "text",      "text": "Describe the contents of this page, including any tables or figures."},
            {"type": "image_url", "image_url": {"url": f"data:{img['mime']};base64,{img['b64']}"}},
        ],
    }],
)
print(response.choices[0].message.content)

### 13c. Extract tables with `pdfplumber` → Markdown

`pdfplumber` uses heuristics to detect table regions. Tables are exported as Markdown and embedded in the text message so the model can reason about them structurally.

In [ ]:
import pdfplumber

def extract_tables_as_markdown(path: str) -> str:
    """Extract all tables from a PDF and return them as Markdown."""
    blocks = []
    with pdfplumber.open(path) as pdf:
        for p_num, page in enumerate(pdf.pages):
            for t_idx, table in enumerate(page.extract_tables()):
                if not table:
                    continue
                header = table[0]
                rows   = table[1:]
                lines  = ["| " + " | ".join(str(c or "") for c in header) + " |",
                          "|" + "|".join("---" for _ in header) + "|"]
                for row in rows:
                    lines.append("| " + " | ".join(str(c or "") for c in row) + " |")
                blocks.append(f"**Table {t_idx+1} (page {p_num+1}):**\n" + "\n".join(lines))
    return "\n\n".join(blocks)

tables_md = extract_tables_as_markdown(pdf_path)
print(tables_md if tables_md else "(no tables detected by pdfplumber — try with a real PDF)")

if tables_md:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a data analyst. Tables are Markdown-formatted."},
            {"role": "user",   "content": f"Analyse the following tables:\n\n{tables_md}"},
        ],
    )
    print("\nAnalysis:", response.choices[0].message.content)

### 13d. Full pipeline — text + page images + tables in one message

Combine all three content types. Cap images at 5 pages to stay within the token budget.

In [ ]:
def build_pdf_message(path: str, question: str, max_pages: int = 5) -> list[dict]:
    """Build a multimodal message from a PDF (text + page images + tables)."""
    text   = pdf_to_text(path)
    tables = extract_tables_as_markdown(path)
    pages  = pdf_pages_as_images(path)

    body = f"{question}\n\n## Document text\n\n{text}"
    if tables:
        body += f"\n\n## Tables (Markdown)\n\n{tables}"

    content: list = [{"type": "text", "text": body}]
    for img in pages[:max_pages]:
        content.append({
            "type": "image_url",
            "image_url": {"url": f"data:{img['mime']};base64,{img['b64']}"},
        })
    return [{"role": "user", "content": content}]

messages = build_pdf_message(
    pdf_path,
    "What are the key findings in this report? Mention any data from figures or tables.",
)
response = client.chat.completions.create(model="gpt-4o-mini", messages=messages)
print(response.choices[0].message.content)

# Clean up
import os
os.unlink(pdf_path)